Create model and export it. For this current notebok, we put in a fake logistic model with random coefficients and an intercept. We then replaced it with our production model values.

In the future, this could be a Data Science notebook, which exports the pickled artifacts at the end.

In [61]:
#Force installation of docker image compatible scikit-learn package
#More current version is not supported

# !pip install scikit-learn==0.24.2 --user
# !pip install pickle5 --user
!pip install scikit-learn==1.3.2 --user


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 97.3 MB/s eta 0:00:00:00:010:01


In [1]:
from sklearn.linear_model import LogisticRegression
import numpy as np


#create fake model (as a test, no training data)
#final_model.coef_=np.array([[0.5,0.1,0.2,0.3]])
# final_model.coef_=np.array([[0.05,0.01,0.02,0.03]])
#final_model.intercept_=-1.0

#artifically create last model (one currently running in prod)
#Inputs: optimal_threshold, lastFivePerf
#projectedAccuracy = 1 / (1 + np.exp(-1*(C+ D * optimal_threshold + E * lastFivePerf)))
#C,D,E =  -5, 3, 6.3
#Note: ranges of [-4.8, 2.97, 6.0] → [-5.0247, 3.1002, 6.3360] worked similarily 


final_model = LogisticRegression()
final_model.coef_=np.array([[ 3.1002, 6.3360 ]])
final_model.intercept_=-5.0247
final_model.classes_=np.array([0, 1])

#check w predictions
data = np.array([[1, 1],[.5, 1],[0, 1], [1,1], [1,.75], [1,.5]])
final_model.predict_proba(data)[:,1]



array([0.98800858, 0.94590498, 0.78773061, 0.98800858, 0.94414391,
       0.77617265])

In [2]:
print(np.shape(final_model.coef_))


(1, 2)


In [55]:
# data = np.array([[1, 1,1,1],[2, 2,2,2],[1.5,1.5,1.5,1.5]])


# print(data)
# # data = data.tobytes()
# data = np.array2string(data, separator=',')
# print(type(data))
# data=data.encode()
# print(type(data))

# import flask
# import io
# import pandas as pd
# data = data.decode("utf-8")
# print(data)

# # data = np.frombuffer(data, dtype=np.float64)
# # print(data)

# #s = io.StringIO(data)
# data= np.array(eval(data))
# data = pd.DataFrame(data)
# print(data)

# # final_model.predict_proba(data)[:,1]


In [56]:
# data

In [57]:
# final_model.predict(x)

In [74]:
pwd

'/home/ec2-user/SageMaker/[REDACTED]AppliedScience/Omar/ModelImplementationWSDK'

In [95]:
modelname = 'model'
#prefix = "./custom_model/container/"+modelname+"/"
prefix = "custom_model/container/"
local_docker_path = ""
# local_docker_path = "local_test/test_dir/model/"
# local_docker_path = "opt/ml/model/"
model_path = prefix+local_docker_path

local_docker_path = "local_test/test_dir/model/"
alternate_model_path = prefix+local_docker_path

# #save in the container so dockerfile can copy it
# modelname = 'custom-logistic-model'
# model_path = "./model_template/container/"

model_filename = modelname + ".pkl"
model_zipname = modelname + ".tar.gz"
#bucket_prefix = "testModel-byo-model"
bucket_prefix = modelname[:]

import os

model_filename = os.path.join(model_path,model_filename)
model_zipname = os.path.join(model_path,model_zipname)
print(model_filename)
print(model_zipname)

#copies to folder to-be-dockerized, and to be zipped from

if not os.path.exists(model_path):
    os.makedirs(model_path)
    print(f"Directory created: {model_path}")
else:
    print(f"Directory already exists: {model_path}")


# import pickle5 as pickle
import pickle
with open(model_filename, 'wb') as handle:
    pickle.dump(final_model, handle, protocol=pickle.HIGHEST_PROTOCOL)


#joblib.dump(final_model, model_filename)
print(f"Model saved as {model_filename}")


#dump zip version too
import tarfile
tar = tarfile.open(model_zipname, "w:gz")
tar.add(model_filename)
tar.close()

print(f"zip file saved as {model_zipname}")

custom_model/container/model.pkl
custom_model/container/model.tar.gz
Directory already exists: custom_model/container/
Model saved as custom_model/container/model.pkl
zip file saved as custom_model/container/model.tar.gz


In [89]:
#test with pickle load
# import pickle5 as pickle
import pickle

# Load the .pkl file
with open(model_filename, "rb") as model_pickle_file:
    loaded_model = pickle.load(model_pickle_file)

#check w predictions
x = np.array([[1, 1,1,1],[2, 2,2,2]])
loaded_model.predict_proba(x)[:,1]


array([0.29110983, 0.31431989])

In [90]:
# #export model artifact as *.joblib file
# import joblib
# joblib.dump(final_model, model_filename)
# print(f"Model saved as {model_filename}")

#import joblib model from s3
# !aws s3 cp s3://aws-ml-blog/artifacts/scikit_learn_bring_your_own_model/model.joblib .

#zip file
!tar czvf $model_zipname $model_filename
print(f"Model tar zipped: {model_filename}")


# Upload the pre-trained model model.tar.gz file to S3
#aws crednentials for model storage
import sagemaker
from sagemaker import get_execution_role
import boto3

region = boto3.Session().region_name
role = get_execution_role()
bucket = sagemaker.Session().default_bucket()


print(f"bucket: {bucket}")

import os

#fObj = open(model_zipname, "rb")
# key = os.path.join(bucket_prefix, model_zipname)
fObj = open(model_filename, "rb")
key = os.path.join(bucket_prefix, model_filename)
boto3.Session().resource("s3").Bucket(bucket).Object(key).upload_fileobj(fObj)

model_data = "s3://{}/{}".format(bucket, key)
print(f"model data: {model_data}")


custom_model/container/model.pkl
Model tar zipped: custom_model/container/model.pkl
bucket: sagemaker-us-east-1-[REDACTED]
model data: s3://sagemaker-us-east-1-[REDACTED]/model/custom_model/container/model.pkl
